# Protein MLM with a Fresh BERT Model

This notebook builds a configurable masked language modeling workflow for protein sequences.

It is designed to:

- reuse the project's saved tokenizer assets
- create or reload tokenized dataset artifacts
- initialize a fresh `BertForMaskedLM` from a downloaded BERT config
- keep training optional until you explicitly enable it in the config block


## Imports

Keep imports isolated so dependency issues surface early and configuration stays uncluttered.


In [ ]:
from collections import deque
from concurrent.futures import ProcessPoolExecutor
import gzip
import hashlib
import inspect
import json
import math
import multiprocessing as mp
import os
import shutil
from pathlib import Path

os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "1"


import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import torch

from tqdm.auto import tqdm
from transformers import BertConfig, BertForMaskedLM
from transformers import DataCollatorForLanguageModeling, PreTrainedTokenizerFast, Trainer, TrainingArguments, set_seed

from util.string_utils import humanizeNumericString

## Configuration

All knobs for paths, tokenization reuse, model shape, and MLM training live here. Keep edits in this section instead of scattering flags across later cells.


In [ ]:
SEED = 42

VISIBLE_GPU_IDS = os.environ.get("CUDA_VISIBLE_DEVICES", "0")

# Tokenizer artifacts
TOKENIZATION_STRATEGY = "BPE"  # ["Unigram", "BPE", "WordPiece", "words", "pairs", "k-mers", "SentencePiece"]
VOCAB_SIZE = 512
RARE_RESIDUE_POLICY = "keep"

MAX_LENGTH = 512
TRAIN_FRACTION = 1.0
VALIDATION_SPLIT = 0.1
ENCODING_BATCH_SIZE = 512
TOKENIZED_DATASET_DIRNAME = "streaming_tokenized_dataset"
TOKENIZED_DATASET_FORMAT = "parquet"  # ["parquet", "jsonl", "jsonl.gz"]
TOKENIZED_DATASET_PARQUET_COMPRESSION = "zstd"
TOKENIZED_DATASET_SHARD_SIZE = 50_000
STREAMING_SHUFFLE_BUFFER_SIZE = 10_000
TOKENIZATION_NUM_WORKERS = max(1, min(8, (os.cpu_count() or 1) - 1))
MAX_PENDING_TOKENIZATION_TASKS = max(1, TOKENIZATION_NUM_WORKERS * 2)
RUN_TOKENIZATION = True
REBUILD_TOKENIZED_DATASET = False

# BERT Config
BASE_MODEL_CONFIG_ID = "bert-base-uncased"
HIDDEN_SIZE = 768
NUM_HIDDEN_LAYERS = 4
INTERMEDIATE_SIZE = 3072
MAX_POSITION_EMBEDDINGS = 512
TYPE_VOCAB_SIZE = 2

MODEL_HIDDEN_SIZE = None
MODEL_NUM_HIDDEN_LAYERS = None
MODEL_NUM_ATTENTION_HEADS = None
MODEL_INTERMEDIATE_SIZE = None
HIDDEN_DROPOUT_PROB = 0.1
ATTENTION_PROBS_DROPOUT_PROB = 0.1

MLM_PROBABILITY = 0.15
NUM_TRAIN_EPOCHS = 1.0
MAX_STEPS = -1
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 16
GRADIENT_ACCUMULATION_STEPS = 1
LEARNING_RATE = 5e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.0
LOGGING_STEPS = 50
EVAL_STEPS = 200
SAVE_STEPS = 200
SAVE_TOTAL_LIMIT = 2
DATALOADER_NUM_WORKERS = 0

EVALUATION_STRATEGY = "steps"
SAVE_STRATEGY = "steps"
LOAD_BEST_MODEL_AT_END = True
OVERWRITE_OUTPUT_DIR = True
USE_FP16 = torch.cuda.is_available()
USE_BF16 = False

TOKENIZER_ARTIFACTS_DIR_RELATIVE_PATH = "results/protein_tokenization/artifacts/tokenizers"
MODEL_OUTPUT_DIR_RELATIVE_PATH = f"results/bert_mlm/{TOKENIZATION_STRATEGY.lower()}_vocab{VOCAB_SIZE}_{RARE_RESIDUE_POLICY}_max{MAX_LENGTH}"

RUN_TRAINING = True
RUN_EVALUATION = False
RESUME_FROM_CHECKPOINT = None

In [ ]:
def bert_param_count_str(vocab_size, hidden_size, num_hidden_layers, intermediate_size, max_position_embeddings, type_vocab_size):
    # Embeddings
    embeddings = vocab_size * hidden_size + max_position_embeddings * hidden_size + type_vocab_size * hidden_size
    # Per layer
    per_layer = 4 * (hidden_size**2) + 2 * hidden_size * intermediate_size
    # Total
    total = embeddings + num_hidden_layers * per_layer
    return humanizeNumericString(total, decimals=0)

## Project Paths and Helpers

This section resolves the tokenizer artifact paths, fails fast when required assets are missing, and loads the saved fast tokenizer plus normalized corpus files.


In [ ]:
def find_project_root(start):
    for candidate in [start] + list(start.parents):
        if (candidate / "package.json").exists() and (candidate / "results").exists():
            return candidate
    raise FileNotFoundError("Could not locate the repository root from the current notebook working directory.")


def ensure_exists(path, label):
    if not path.exists():
        raise FileNotFoundError(f"{label} not found: {path}")


PROJECT_ROOT = find_project_root(Path.cwd())
TOKENIZER_ARTIFACTS_DIR = PROJECT_ROOT / TOKENIZER_ARTIFACTS_DIR_RELATIVE_PATH
TOKENIZER_RUN_DIR = TOKENIZER_ARTIFACTS_DIR / TOKENIZATION_STRATEGY / f"vocab{VOCAB_SIZE}_{RARE_RESIDUE_POLICY}"
FAST_TOKENIZER_DIR = TOKENIZER_RUN_DIR / f"{TOKENIZATION_STRATEGY}_fast_tokenizer"
NORMALIZED_CORPUS_DIR = TOKENIZER_RUN_DIR / "normalized_corpus"
TOKENIZER_JSON_PATH = FAST_TOKENIZER_DIR / "tokenizer.json"
TOKENIZER_CONFIG_PATH = FAST_TOKENIZER_DIR / "tokenizer_config.json"
NORMALIZED_CORPUS_FILES = sorted(NORMALIZED_CORPUS_DIR.glob("*.txt"))
MODEL_OUTPUT_DIR = PROJECT_ROOT / MODEL_OUTPUT_DIR_RELATIVE_PATH
TOKENIZED_DATASET_DIR = MODEL_OUTPUT_DIR / TOKENIZED_DATASET_DIRNAME
TOKENIZED_DATASET_METADATA_PATH = TOKENIZED_DATASET_DIR / "metadata.json"

ensure_exists(TOKENIZER_ARTIFACTS_DIR, "Tokenizer artifacts directory")
ensure_exists(TOKENIZER_RUN_DIR, "Tokenizer run directory")
ensure_exists(FAST_TOKENIZER_DIR, "Fast tokenizer directory")
ensure_exists(TOKENIZER_JSON_PATH, "Fast tokenizer JSON")
ensure_exists(TOKENIZER_CONFIG_PATH, "Fast tokenizer config")
ensure_exists(NORMALIZED_CORPUS_DIR, "Normalized corpus directory")

if not NORMALIZED_CORPUS_FILES:
    raise FileNotFoundError(f"No normalized corpus text files found in: {NORMALIZED_CORPUS_DIR}")

set_seed(SEED)

tokenizer = PreTrainedTokenizerFast.from_pretrained(str(FAST_TOKENIZER_DIR))
tokenizer.model_max_length = MAX_LENGTH

missing_special_tokens = []
for token_name in ["pad_token", "unk_token", "cls_token", "sep_token", "mask_token"]:
    if getattr(tokenizer, token_name) is None:
        missing_special_tokens.append(token_name)
if missing_special_tokens:
    raise ValueError(f"Tokenizer is missing required special tokens: {missing_special_tokens}")

print(f"Loaded fast tokenizer from: {FAST_TOKENIZER_DIR}")
print(f"Found {len(NORMALIZED_CORPUS_FILES):,} normalized corpus files in: {NORMALIZED_CORPUS_DIR}")

pd.DataFrame(
    [
        {"artifact": "tokenizer_run_dir", "path": str(TOKENIZER_RUN_DIR), "exists": TOKENIZER_RUN_DIR.exists()},
        {"artifact": "fast_tokenizer_dir", "path": str(FAST_TOKENIZER_DIR), "exists": FAST_TOKENIZER_DIR.exists()},
        {"artifact": "normalized_corpus_dir", "path": str(NORMALIZED_CORPUS_DIR), "exists": NORMALIZED_CORPUS_DIR.exists()},
        {"artifact": "normalized_corpus_files", "path": str(NORMALIZED_CORPUS_DIR), "exists": len(NORMALIZED_CORPUS_FILES) > 0, "count": len(NORMALIZED_CORPUS_FILES)},
        {"artifact": "model_output_dir", "path": str(MODEL_OUTPUT_DIR), "exists": MODEL_OUTPUT_DIR.exists()},
        {"artifact": "tokenized_dataset_dir", "path": str(TOKENIZED_DATASET_DIR), "exists": TOKENIZED_DATASET_DIR.exists()},
        {"artifact": "tokenized_dataset_metadata", "path": str(TOKENIZED_DATASET_METADATA_PATH), "exists": TOKENIZED_DATASET_METADATA_PATH.exists()},
    ]
)

## Load and Encode the Normalized Corpus

This section reads the pre-tokenized normalized corpus from the tokenizer artifacts and converts each whitespace-separated row into MLM-ready token IDs. If the corpus assets are missing, the notebook fails immediately.


In [ ]:
_WORKER_TOKENIZER = None


def stable_fraction(*parts):
    digest = hashlib.blake2b("::".join(str(part) for part in parts).encode("utf-8"), digest_size=8).digest()
    return int.from_bytes(digest, byteorder="big") / float(1 << 64)


def get_tokenized_dataset_suffix(dataset_format):
    if dataset_format == "parquet":
        return ".parquet"
    if dataset_format == "jsonl":
        return ".jsonl"
    if dataset_format == "jsonl.gz":
        return ".jsonl.gz"
    raise ValueError("TOKENIZED_DATASET_FORMAT must be 'parquet', 'jsonl', or 'jsonl.gz'.")


def open_tokenized_file(file_path, mode, dataset_format):
    if dataset_format == "jsonl":
        return file_path.open(mode, encoding="utf-8")
    if dataset_format == "jsonl.gz":
        return gzip.open(file_path, mode=mode, encoding="utf-8")
    raise ValueError(f"Unsupported text tokenized dataset format: {dataset_format}")


def encode_batch_with_tokenizer(tokenizer_instance, text_batch):
    token_rows = [text.strip().split() for text in text_batch]
    encoded = tokenizer_instance(
        token_rows,
        is_split_into_words=True,
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
        return_special_tokens_mask=True,
    )

    batch_records = []
    encoded_dict = dict(encoded)
    batch_size = len(text_batch)
    for row_index in range(batch_size):
        record = {}
        for key, value in encoded_dict.items():
            record[key] = value[row_index]
        batch_records.append(record)

    return batch_records


def tokenize_pretokenized_batch(text_batch):
    return encode_batch_with_tokenizer(tokenizer, text_batch)


def init_tokenizer_worker(fast_tokenizer_dir, max_length):
    global _WORKER_TOKENIZER
    _WORKER_TOKENIZER = PreTrainedTokenizerFast.from_pretrained(fast_tokenizer_dir)
    _WORKER_TOKENIZER.model_max_length = max_length


def tokenize_pretokenized_batch_in_worker(text_batch):
    if _WORKER_TOKENIZER is None:
        raise RuntimeError("Worker tokenizer is not initialized.")
    return encode_batch_with_tokenizer(_WORKER_TOKENIZER, text_batch)


def iter_corpus_rows():
    file_iterator = tqdm(NORMALIZED_CORPUS_FILES, desc="Scanning corpus files", unit="file")
    for file_path in file_iterator:
        with file_path.open("r", encoding="utf-8") as handle:
            for line_number, line in enumerate(handle, start=1):
                normalized_text = line.strip()
                if not normalized_text:
                    continue

                row_id = f"{file_path.name}:{line_number}"
                if stable_fraction("train_fraction", row_id) >= TRAIN_FRACTION:
                    continue

                split_name = "train"
                if VALIDATION_SPLIT > 0 and stable_fraction("validation_split", row_id) < VALIDATION_SPLIT:
                    split_name = "validation"

                yield row_id, split_name, normalized_text


def iter_corpus_batches(batch_size):
    batch = []
    for row_id, split_name, normalized_text in iter_corpus_rows():
        batch.append((row_id, split_name, normalized_text))
        if len(batch) >= batch_size:
            yield batch
            batch = []

    if batch:
        yield batch


def write_jsonl_record(handle, record):
    handle.write(json.dumps(record))
    handle.write("\n")


def group_records_by_split(batch, tokenized_records):
    records_by_split = {"train": [], "validation": []}
    for (row_id, split_name, _), tokenized_record in zip(batch, tokenized_records):
        tokenized_record["row_id"] = row_id
        records_by_split[split_name].append(tokenized_record)
    return records_by_split


def write_records_to_text_shards(records_by_split, split_handles, split_rows_in_shard, split_shard_index, split_files, split_counts, split_dirs, dataset_format):
    def open_next_shard(split_name):
        if split_handles[split_name] is not None:
            split_handles[split_name].close()

        shard_suffix = get_tokenized_dataset_suffix(dataset_format)
        shard_path = split_dirs[split_name] / f"part-{split_shard_index[split_name]:05d}{shard_suffix}"
        split_shard_index[split_name] += 1
        split_files[split_name].append(shard_path)
        split_rows_in_shard[split_name] = 0
        split_handles[split_name] = open_tokenized_file(shard_path, mode="wt", dataset_format=dataset_format)

    for split_name, split_records in records_by_split.items():
        if not split_records:
            continue
        if split_handles[split_name] is None or split_rows_in_shard[split_name] + len(split_records) > TOKENIZED_DATASET_SHARD_SIZE:
            open_next_shard(split_name)

        for record in split_records:
            write_jsonl_record(split_handles[split_name], record)

        split_counts[split_name] += len(split_records)
        split_rows_in_shard[split_name] += len(split_records)


def write_records_to_parquet_shards(records_by_split, split_writers, split_rows_in_shard, split_shard_index, split_files, split_counts, split_dirs):
    def open_next_shard(split_name, table):
        if split_writers[split_name] is not None:
            split_writers[split_name].close()

        shard_path = split_dirs[split_name] / f"part-{split_shard_index[split_name]:05d}.parquet"
        split_shard_index[split_name] += 1
        split_files[split_name].append(shard_path)
        split_rows_in_shard[split_name] = 0
        split_writers[split_name] = pq.ParquetWriter(
            shard_path,
            table.schema,
            compression=TOKENIZED_DATASET_PARQUET_COMPRESSION,
        )

    for split_name, split_records in records_by_split.items():
        if not split_records:
            continue

        table = pa.Table.from_pylist(split_records)
        if split_writers[split_name] is None or split_rows_in_shard[split_name] + table.num_rows > TOKENIZED_DATASET_SHARD_SIZE:
            open_next_shard(split_name, table)

        split_writers[split_name].write_table(table)
        split_counts[split_name] += table.num_rows
        split_rows_in_shard[split_name] += table.num_rows


def build_tokenized_dataset_on_disk():
    if not 0 < TRAIN_FRACTION <= 1:
        raise ValueError("TRAIN_FRACTION must be in (0, 1].")
    if not 0 <= VALIDATION_SPLIT < 1:
        raise ValueError("VALIDATION_SPLIT must be in [0, 1).")
    if ENCODING_BATCH_SIZE <= 0:
        raise ValueError("ENCODING_BATCH_SIZE must be a positive integer.")
    if TOKENIZATION_NUM_WORKERS <= 0:
        raise ValueError("TOKENIZATION_NUM_WORKERS must be a positive integer.")
    if MAX_PENDING_TOKENIZATION_TASKS <= 0:
        raise ValueError("MAX_PENDING_TOKENIZATION_TASKS must be a positive integer.")

    dataset_format = TOKENIZED_DATASET_FORMAT
    get_tokenized_dataset_suffix(dataset_format)

    if REBUILD_TOKENIZED_DATASET and not RUN_TOKENIZATION:
        raise ValueError("REBUILD_TOKENIZED_DATASET requires RUN_TOKENIZATION = True.")

    if TOKENIZED_DATASET_DIR.exists() and not TOKENIZED_DATASET_METADATA_PATH.exists():
        print(f"Removing incomplete tokenized dataset directory: {TOKENIZED_DATASET_DIR}")
        shutil.rmtree(TOKENIZED_DATASET_DIR)

    if REBUILD_TOKENIZED_DATASET and TOKENIZED_DATASET_DIR.exists():
        shutil.rmtree(TOKENIZED_DATASET_DIR)

    TOKENIZED_DATASET_DIR.mkdir(parents=True, exist_ok=True)

    split_dirs = {
        "train": TOKENIZED_DATASET_DIR / "train",
        "validation": TOKENIZED_DATASET_DIR / "validation",
    }
    for split_dir in split_dirs.values():
        split_dir.mkdir(parents=True, exist_ok=True)

    split_counts = {"train": 0, "validation": 0}
    split_files = {"train": [], "validation": []}
    split_rows_in_shard = {"train": 0, "validation": 0}
    split_shard_index = {"train": 0, "validation": 0}
    split_handles = {"train": None, "validation": None}
    split_writers = {"train": None, "validation": None}

    row_progress = tqdm(desc="Tokenizing rows to disk", unit="row")
    pending_tasks = deque()

    def persist_tokenized_batch(batch, tokenized_records):
        records_by_split = group_records_by_split(batch, tokenized_records)
        if dataset_format == "parquet":
            write_records_to_parquet_shards(
                records_by_split,
                split_writers,
                split_rows_in_shard,
                split_shard_index,
                split_files,
                split_counts,
                split_dirs,
            )
        else:
            write_records_to_text_shards(
                records_by_split,
                split_handles,
                split_rows_in_shard,
                split_shard_index,
                split_files,
                split_counts,
                split_dirs,
                dataset_format,
            )
        row_progress.update(len(batch))
        row_progress.set_postfix(train=split_counts["train"], validation=split_counts["validation"])

    def flush_oldest_task():
        batch, future = pending_tasks.popleft()
        tokenized_records = future.result()
        persist_tokenized_batch(batch, tokenized_records)

    try:
        if TOKENIZATION_NUM_WORKERS == 1:
            for batch in iter_corpus_batches(ENCODING_BATCH_SIZE):
                tokenized_records = tokenize_pretokenized_batch([normalized_text for _, _, normalized_text in batch])
                persist_tokenized_batch(batch, tokenized_records)
        else:
            with ProcessPoolExecutor(
                max_workers=TOKENIZATION_NUM_WORKERS,
                mp_context=mp.get_context("fork"),
                initializer=init_tokenizer_worker,
                initargs=(str(FAST_TOKENIZER_DIR), MAX_LENGTH),
            ) as executor:
                for batch in iter_corpus_batches(ENCODING_BATCH_SIZE):
                    text_batch = [normalized_text for _, _, normalized_text in batch]
                    future = executor.submit(tokenize_pretokenized_batch_in_worker, text_batch)
                    pending_tasks.append((batch, future))

                    while len(pending_tasks) >= MAX_PENDING_TOKENIZATION_TASKS:
                        flush_oldest_task()

                while pending_tasks:
                    flush_oldest_task()
    except Exception:
        if TOKENIZED_DATASET_DIR.exists() and not TOKENIZED_DATASET_METADATA_PATH.exists():
            print(f"Cleaning failed tokenized dataset build at: {TOKENIZED_DATASET_DIR}")
            shutil.rmtree(TOKENIZED_DATASET_DIR, ignore_errors=True)
        raise
    finally:
        row_progress.close()
        for split_name, handle in split_handles.items():
            if handle is not None:
                handle.close()
        for split_name, writer in split_writers.items():
            if writer is not None:
                writer.close()

    if split_counts["train"] == 0:
        raise ValueError("No normalized corpus rows were assigned to the training split.")

    metadata = {
        "format": dataset_format,
        "tokenized_dataset_dir": str(TOKENIZED_DATASET_DIR),
        "split_counts": split_counts,
        "data_files": {split_name: [str(path) for path in paths] for split_name, paths in split_files.items()},
        "train_fraction": TRAIN_FRACTION,
        "validation_split": VALIDATION_SPLIT,
        "encoding_batch_size": ENCODING_BATCH_SIZE,
        "tokenization_num_workers": TOKENIZATION_NUM_WORKERS,
        "parquet_compression": TOKENIZED_DATASET_PARQUET_COMPRESSION if dataset_format == "parquet" else None,
        "max_length": MAX_LENGTH,
        "vocab_size": len(tokenizer),
    }

    TOKENIZED_DATASET_METADATA_PATH.write_text(json.dumps(metadata, indent=2), encoding="utf-8")
    return metadata


def buffered_shuffle(iterator, buffer_size, seed):
    rng = np.random.default_rng(seed)
    buffer = []

    for item in iterator:
        buffer.append(item)
        if len(buffer) < buffer_size:
            continue

        index = int(rng.integers(0, len(buffer)))
        yield buffer.pop(index)

    while buffer:
        index = int(rng.integers(0, len(buffer)))
        yield buffer.pop(index)


class DiskBackedTokenizedSplit(torch.utils.data.IterableDataset):
    def __init__(self, file_paths, row_count, dataset_format, shuffle=False, shuffle_buffer_size=0, seed=SEED):
        self.file_paths = [Path(file_path) for file_path in file_paths]
        self.row_count = row_count
        self.dataset_format = dataset_format
        self.shuffle = shuffle
        self.shuffle_buffer_size = shuffle_buffer_size
        self.seed = seed
        self.epoch = 0

    def set_epoch(self, epoch):
        self.epoch = epoch

    def __len__(self):
        return self.row_count

    def _iter_parquet_records(self):
        for file_path in self.file_paths:
            parquet_file = pq.ParquetFile(file_path)
            for record_batch in parquet_file.iter_batches(batch_size=ENCODING_BATCH_SIZE):
                table = pa.Table.from_batches([record_batch])
                for record in table.to_pylist():
                    record.pop("row_id", None)
                    yield record

    def _iter_text_records(self):
        for file_path in self.file_paths:
            with open_tokenized_file(file_path, mode="rt", dataset_format=self.dataset_format) as handle:
                for line in handle:
                    if not line.strip():
                        continue
                    record = json.loads(line)
                    record.pop("row_id", None)
                    yield record

    def _iter_records(self):
        if self.dataset_format == "parquet":
            yield from self._iter_parquet_records()
            return

        yield from self._iter_text_records()

    def __iter__(self):
        iterator = self._iter_records()
        if self.shuffle and self.shuffle_buffer_size > 1 and self.row_count > 1:
            yield from buffered_shuffle(iterator, min(self.shuffle_buffer_size, self.row_count), self.seed + self.epoch)
            return

        yield from iterator


def load_tokenized_dataset_metadata():
    if TOKENIZED_DATASET_METADATA_PATH.exists() and not REBUILD_TOKENIZED_DATASET:
        print(f"Using existing tokenized dataset from: {TOKENIZED_DATASET_DIR}")
        return json.loads(TOKENIZED_DATASET_METADATA_PATH.read_text(encoding="utf-8"))

    if not RUN_TOKENIZATION:
        raise FileNotFoundError("Tokenized dataset metadata was not found and RUN_TOKENIZATION is False. Enable RUN_TOKENIZATION or point the notebook at an existing tokenized dataset.")

    print(f"Building tokenized dataset at: {TOKENIZED_DATASET_DIR}")
    return build_tokenized_dataset_on_disk()


def load_disk_backed_tokenized_dataset():
    metadata = load_tokenized_dataset_metadata()
    dataset_format = metadata.get("format", "jsonl.gz")

    dataset = {
        "train": DiskBackedTokenizedSplit(
            metadata["data_files"]["train"],
            metadata["split_counts"]["train"],
            dataset_format=dataset_format,
            shuffle=True,
            shuffle_buffer_size=STREAMING_SHUFFLE_BUFFER_SIZE,
            seed=SEED,
        )
    }

    if metadata["split_counts"].get("validation", 0) > 0:
        dataset["validation"] = DiskBackedTokenizedSplit(
            metadata["data_files"]["validation"],
            metadata["split_counts"]["validation"],
            dataset_format=dataset_format,
            shuffle=False,
            seed=SEED,
        )

    return dataset, metadata


def load_first_tokenized_record(split_name, metadata):
    split_files = metadata["data_files"].get(split_name, [])
    if not split_files:
        raise ValueError(f"No files found for split: {split_name}")

    dataset_format = metadata.get("format", "jsonl.gz")
    first_path = Path(split_files[0])

    if dataset_format == "parquet":
        parquet_file = pq.ParquetFile(first_path)
        for record_batch in parquet_file.iter_batches(batch_size=1):
            table = pa.Table.from_batches([record_batch])
            records = table.to_pylist()
            if records:
                record = records[0]
                record.pop("row_id", None)
                return record
    else:
        with open_tokenized_file(first_path, mode="rt", dataset_format=dataset_format) as handle:
            for line in handle:
                if line.strip():
                    record = json.loads(line)
                    record.pop("row_id", None)
                    return record

    raise ValueError(f"No records found for split: {split_name}")


tokenized_dataset, tokenized_dataset_metadata = load_disk_backed_tokenized_dataset()

pd.DataFrame(
    [
        {
            "split": split_name,
            "rows": tokenized_dataset_metadata["split_counts"][split_name],
            "shards": len(tokenized_dataset_metadata["data_files"].get(split_name, [])),
            "encoding_batch_size": tokenized_dataset_metadata.get("encoding_batch_size", ENCODING_BATCH_SIZE),
            "tokenization_num_workers": tokenized_dataset_metadata.get("tokenization_num_workers", 1),
            "format": tokenized_dataset_metadata["format"],
        }
        for split_name in tokenized_dataset
    ]
)

In [ ]:
pd.Series(
    {
        "tokenized_dataset_dir": str(TOKENIZED_DATASET_DIR),
        "tokenized_dataset_metadata": str(TOKENIZED_DATASET_METADATA_PATH),
        "tokenized_dataset_format": TOKENIZED_DATASET_FORMAT,
        "parquet_compression": TOKENIZED_DATASET_PARQUET_COMPRESSION if TOKENIZED_DATASET_FORMAT == "parquet" else None,
        "encoding_batch_size": ENCODING_BATCH_SIZE,
        "tokenization_num_workers": TOKENIZATION_NUM_WORKERS,
        "max_pending_tokenization_tasks": MAX_PENDING_TOKENIZATION_TASKS,
        "run_tokenization": RUN_TOKENIZATION,
        "rebuild_tokenized_dataset": REBUILD_TOKENIZED_DATASET,
    }
)

## Inspect Tokenized Samples

Use this section to verify that the cached token IDs still map cleanly back to the saved tokenizer vocabulary.


In [ ]:
sample_split_name = "validation" if "validation" in tokenized_dataset else "train"
sample_record = load_first_tokenized_record(sample_split_name, tokenized_dataset_metadata)
sample_tokens = tokenizer.convert_ids_to_tokens(sample_record["input_ids"])

pd.DataFrame(
    {
        "token_id": sample_record["input_ids"],
        "token": sample_tokens,
        "special_token_mask": sample_record["special_tokens_mask"],
    }
).head(32)

## Initialize a Fresh BERT MLM Model

This downloads the configuration template from Hugging Face and instantiates a new `BertForMaskedLM` with random weights sized to the local tokenizer vocabulary.


In [ ]:
def build_untrained_mlm_model():
    config = BertConfig.from_pretrained(BASE_MODEL_CONFIG_ID)

    overrides = {
        "hidden_size": MODEL_HIDDEN_SIZE,
        "num_hidden_layers": MODEL_NUM_HIDDEN_LAYERS,
        "num_attention_heads": MODEL_NUM_ATTENTION_HEADS,
        "intermediate_size": MODEL_INTERMEDIATE_SIZE,
    }
    for field_name, value in overrides.items():
        if value is not None:
            setattr(config, field_name, value)

    config.vocab_size = len(tokenizer)
    config.max_position_embeddings = max(config.max_position_embeddings, MAX_LENGTH + 2)
    config.hidden_dropout_prob = HIDDEN_DROPOUT_PROB
    config.attention_probs_dropout_prob = ATTENTION_PROBS_DROPOUT_PROB
    config.pad_token_id = tokenizer.pad_token_id
    config.bos_token_id = tokenizer.cls_token_id
    config.eos_token_id = tokenizer.sep_token_id

    if config.hidden_size % config.num_attention_heads != 0:
        raise ValueError("MODEL_HIDDEN_SIZE must be divisible by MODEL_NUM_ATTENTION_HEADS.")

    model = BertForMaskedLM(config)
    model.resize_token_embeddings(len(tokenizer))
    return model


model = build_untrained_mlm_model()
trainable_parameters = 0
for parameter in model.parameters():
    if parameter.requires_grad:
        trainable_parameters += parameter.numel()

pd.Series(
    {
        "config_source": BASE_MODEL_CONFIG_ID,
        "fresh_random_weights": True,
        "vocab_size": model.config.vocab_size,
        "max_position_embeddings": model.config.max_position_embeddings,
        "hidden_size": model.config.hidden_size,
        "num_hidden_layers": model.config.num_hidden_layers,
        "num_attention_heads": model.config.num_attention_heads,
        "trainable_parameters": trainable_parameters,
    }
)

## Configure the MLM Trainer

The trainer is assembled here, but training stays disabled unless you switch `RUN_TRAINING` on in the configuration block.


In [ ]:
MODEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

run_evaluation = RUN_EVALUATION and "validation" in tokenized_dataset
save_strategy = SAVE_STRATEGY if RUN_TRAINING else "no"
eval_strategy = EVALUATION_STRATEGY if run_evaluation else "no"
load_best_model_at_end = LOAD_BEST_MODEL_AT_END and RUN_TRAINING and run_evaluation

if LOAD_BEST_MODEL_AT_END and not load_best_model_at_end:
    print("LOAD_BEST_MODEL_AT_END was disabled because evaluation is not active.")

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=MLM_PROBABILITY,
    pad_to_multiple_of=8 if USE_FP16 else None,
)

training_argument_values = {
    "output_dir": str(MODEL_OUTPUT_DIR),
    "overwrite_output_dir": OVERWRITE_OUTPUT_DIR,
    "do_train": RUN_TRAINING,
    "do_eval": run_evaluation,
    "eval_strategy": eval_strategy,
    "evaluation_strategy": eval_strategy,
    "save_strategy": save_strategy,
    "logging_strategy": "steps",
    "logging_steps": LOGGING_STEPS,
    "per_device_train_batch_size": TRAIN_BATCH_SIZE,
    "per_device_eval_batch_size": EVAL_BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "warmup_ratio": WARMUP_RATIO,
    "num_train_epochs": NUM_TRAIN_EPOCHS,
    "max_steps": MAX_STEPS,
    "eval_steps": EVAL_STEPS if run_evaluation else None,
    "save_steps": SAVE_STEPS if RUN_TRAINING else None,
    "save_total_limit": SAVE_TOTAL_LIMIT,
    "load_best_model_at_end": load_best_model_at_end,
    "metric_for_best_model": "eval_loss" if load_best_model_at_end else None,
    "greater_is_better": False if load_best_model_at_end else None,
    "report_to": [],
    "remove_unused_columns": False,
    "dataloader_num_workers": DATALOADER_NUM_WORKERS,
    "seed": SEED,
    "fp16": USE_FP16,
    "bf16": USE_BF16,
}

training_argument_signature = inspect.signature(TrainingArguments.__init__).parameters
if "eval_strategy" in training_argument_signature:
    training_argument_values.pop("evaluation_strategy", None)
elif "evaluation_strategy" in training_argument_signature:
    training_argument_values.pop("eval_strategy", None)

supported_training_argument_values = {}
skipped_training_argument_names = []
for name, value in training_argument_values.items():
    if name not in training_argument_signature:
        skipped_training_argument_names.append(name)
        continue
    if value is None:
        continue
    supported_training_argument_values[name] = value

if skipped_training_argument_names:
    print(
        "Skipping unsupported TrainingArguments fields:",
        ", ".join(skipped_training_argument_names),
    )

training_args = TrainingArguments(**supported_training_argument_values)

trainer_values = {
    "model": model,
    "args": training_args,
    "train_dataset": tokenized_dataset["train"],
    "eval_dataset": tokenized_dataset["validation"] if run_evaluation else None,
    "data_collator": data_collator,
    "processing_class": tokenizer,
    "tokenizer": tokenizer,
}
trainer_signature = inspect.signature(Trainer.__init__).parameters

if "processing_class" in trainer_signature:
    trainer_values.pop("tokenizer", None)
elif "tokenizer" in trainer_signature:
    trainer_values.pop("processing_class", None)

supported_trainer_values = {}
for name, value in trainer_values.items():
    if name not in trainer_signature:
        continue
    if value is None:
        continue
    supported_trainer_values[name] = value

trainer = Trainer(**supported_trainer_values)

pd.Series(
    {
        "run_training": RUN_TRAINING,
        "run_evaluation": run_evaluation,
        "load_best_model_at_end": load_best_model_at_end,
        "train_rows": tokenized_dataset_metadata["split_counts"]["train"],
        "validation_rows": tokenized_dataset_metadata["split_counts"].get("validation", 0),
        "mlm_probability": MLM_PROBABILITY,
        "train_batch_size": TRAIN_BATCH_SIZE,
        "eval_batch_size": EVAL_BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "num_train_epochs": NUM_TRAIN_EPOCHS,
        "output_dir": str(MODEL_OUTPUT_DIR),
        "tokenized_dataset_dir": str(TOKENIZED_DATASET_DIR),
    }
)

## Optional Training and Evaluation

This cell is intentionally inert until `RUN_TRAINING` is set to `True`. It saves the trainer state, model weights, tokenizer copy, and evaluation metrics when enabled.


In [ ]:
if RUN_TRAINING:
    train_result = trainer.train(resume_from_checkpoint=RESUME_FROM_CHECKPOINT)
    trainer.save_model()
    tokenizer.save_pretrained(str(MODEL_OUTPUT_DIR / "tokenizer"))
    trainer.save_state()
    trainer.log_metrics("train", train_result.metrics)
    trainer.save_metrics("train", train_result.metrics)

    if run_evaluation:
        eval_metrics = trainer.evaluate()
        if "eval_loss" in eval_metrics:
            eval_metrics["eval_perplexity"] = math.exp(eval_metrics["eval_loss"])
        trainer.log_metrics("eval", eval_metrics)
        trainer.save_metrics("eval", eval_metrics)
        pd.Series(eval_metrics)
    else:
        pd.Series(train_result.metrics)
else:
    print("Trainer is configured but disabled.")
    print("Set RUN_TRAINING = True in the configuration cell to launch MLM training.")
    print(f"Fast tokenizer directory: {FAST_TOKENIZER_DIR}")
    print(f"Normalized corpus directory: {NORMALIZED_CORPUS_DIR}")
    print(f"Model output directory: {MODEL_OUTPUT_DIR}")

## Artifact and Run Summary

This section records the tokenizer artifact paths, normalized corpus source, and active training configuration so each run leaves a lightweight trail of what the notebook was set up to do.


In [ ]:
run_manifest = {
    "seed": SEED,
    "tokenization_strategy": TOKENIZATION_STRATEGY,
    "tokenizer_vocab_size": VOCAB_SIZE,
    "rare_residue_policy": RARE_RESIDUE_POLICY,
    "tokenizer_run_dir": str(TOKENIZER_RUN_DIR),
    "fast_tokenizer_dir": str(FAST_TOKENIZER_DIR),
    "normalized_corpus_dir": str(NORMALIZED_CORPUS_DIR),
    "normalized_corpus_files": len(NORMALIZED_CORPUS_FILES),
    "model_output_dir": str(MODEL_OUTPUT_DIR),
    "tokenized_dataset_dir": str(TOKENIZED_DATASET_DIR),
    "tokenized_dataset_format": TOKENIZED_DATASET_FORMAT,
    "tokenized_dataset_parquet_compression": TOKENIZED_DATASET_PARQUET_COMPRESSION if TOKENIZED_DATASET_FORMAT == "parquet" else None,
    "tokenization_num_workers": TOKENIZATION_NUM_WORKERS,
    "base_model_config_id": BASE_MODEL_CONFIG_ID,
    "max_length": MAX_LENGTH,
    "validation_split": VALIDATION_SPLIT,
    "train_fraction": TRAIN_FRACTION,
    "encoding_batch_size": ENCODING_BATCH_SIZE,
    "run_tokenization": RUN_TOKENIZATION,
    "rebuild_tokenized_dataset": REBUILD_TOKENIZED_DATASET,
    "run_training": RUN_TRAINING,
    "run_evaluation": RUN_EVALUATION,
    "mlm_probability": MLM_PROBABILITY,
}

manifest_path = MODEL_OUTPUT_DIR / "run_config.json"
manifest_path.write_text(json.dumps(run_manifest, indent=2), encoding="utf-8")

pd.DataFrame(
    [
        {
            "artifact": "tokenizer_run_dir",
            "path": str(TOKENIZER_RUN_DIR),
            "exists": TOKENIZER_RUN_DIR.exists(),
        },
        {
            "artifact": "fast_tokenizer_dir",
            "path": str(FAST_TOKENIZER_DIR),
            "exists": FAST_TOKENIZER_DIR.exists(),
        },
        {
            "artifact": "normalized_corpus_dir",
            "path": str(NORMALIZED_CORPUS_DIR),
            "exists": NORMALIZED_CORPUS_DIR.exists(),
        },
        {
            "artifact": "normalized_corpus_files",
            "path": str(NORMALIZED_CORPUS_DIR),
            "exists": len(NORMALIZED_CORPUS_FILES) > 0,
        },
        {
            "artifact": "tokenized_dataset_dir",
            "path": str(TOKENIZED_DATASET_DIR),
            "exists": TOKENIZED_DATASET_DIR.exists(),
        },
        {
            "artifact": "model_output_dir",
            "path": str(MODEL_OUTPUT_DIR),
            "exists": MODEL_OUTPUT_DIR.exists(),
        },
        {
            "artifact": "run_manifest",
            "path": str(manifest_path),
            "exists": manifest_path.exists(),
        },
    ]
)